# Addison, TX — Land Value Tax Model

**Model type**: Split-rate 4:1 (land taxed at 4× the improvement rate)  
**Levy scope**: City of Addison only  
**Data source**: Dallas Central Appraisal District (DCAD) — Property/ParcelQuery MapServer, Layer 4  
**Tax year**: FY2025 rate ($0.6081/$100); ACFR FY2025 used for revenue validation  

## Column Mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `LNDVALUE` | DCAD certified market value for land |
| Improvement value | `IMPVALUE` | DCAD certified market value for improvements |
| Total market value | `CNTASSDVAL` | = LNDVALUE + IMPVALUE (pre-exemption) |
| Class code | `CLASSCD` | DCAD property class (1=SFR, 17=Commercial, etc.) |
| City filter | `PSTLCITY LIKE '%ADDISON%'` | Postal city |
| BPP exclusion | `CLASSCD NOT IN ('31','32','34','46','33')` | Excludes Business Personal Property |

**Assessment ratio**: 100% — Texas appraises at full market value  
**Millage source**: FY2025 ACFR confirmed rate — $0.6081 per $100 of assessed value  
**Homestead exemption**: 20% off appraised value for owner-occupied residential (city levy); approximated by applying to all CLASS 1/2/3/6 parcels since DCAD MapServer does not expose HS flags  
**Full exemptions**: Identified via owner name matching (government: Town of Addison; educational: Greenhill School; religious: Whiterock Chapel)

## FY2025 ACFR Validation Baseline

| | FY2025 ACFR |
|---|---|
| Real property assessed (gross) | $6,425,521,500 |
| Personal property assessed | $968,200,490 |
| Less tax-exempt property | ($934,769,320) |
| **Total taxable value** | **$6,458,952,670** |
| Adopted tax rate | $0.6081 per $100 |
| Total levy | $39,493,427 |
| Implied real property levy | ~$33.4M |
| Implied BPP levy | ~$5.9M |

**Data vintage note**: ACFR FY2025 levy is based on January 1, 2024 certified values. DCAD MapServer reflects current appraisal (January 1, 2025 values). Our modeled taxable base ($4.5B) is lower than ACFR real property taxable ($5.5B); primary cause is the `PSTLCITY` filter excluding border parcels within Addison city limits that carry a Dallas or Farmers Branch postal city designation.

In [1]:
import sys
import os
import requests
from pathlib import Path
from shapely.geometry import Polygon

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'addison'
STATE_FIPS = '48'    # Texas
COUNTY_FIPS = '113'  # Dallas County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

# FY2026 adopted rate (per $100 assessed value)
MILLAGE_ADDISON = 0.608100

# Homestead exemption: 20% off market value for owner-occupied residential (city levy)
# Applied to CLASS 1 (SFR), 2 (Townhouses), 3 (Condos), 6 (Duplexes) as proxy
# since DCAD MapServer does not expose per-parcel HS flags
HOMESTEAD_RATE = 0.20

# Owner names whose parcels are fully exempt from city taxes
EXEMPT_OWNER_KEYWORDS = [
    'TOWN OF ADDISON', 'ADDISON TOWN OF', 'ADDISON CITY OF',
    'TOWN OF ADDSION',  # typo variant in data
    'GREENHILL SCHOOL',
    'WHITEROCK CHAPEL',
]

# CLASSCD values excluded (Business Personal Property and related)
BPP_CLASSES = ('31', '32', '33', '34', '46')

# Residential classes eligible for homestead exemption approximation
HOMESTEAD_CLASSES = ('1', '2', '3', '6')

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Fetch / Load Parcel Data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
DCAD_QUERY_URL = 'https://maps.dcad.org/prdwa/rest/services/Property/ParcelQuery/MapServer/4/query'

def _download_dcad_addison():
    bpp_list = "','".join(BPP_CLASSES)
    where = f"PSTLCITY LIKE '%ADDISON%' AND CLASSCD NOT IN ('{bpp_list}')"

    count_resp = requests.get(DCAD_QUERY_URL, params={'f': 'json', 'where': where, 'returnCountOnly': 'true'})
    total = count_resp.json().get('count', 0)
    print(f'Total Addison real-property parcels on server: {total:,}')

    all_features = []
    offset = 0
    chunk = 4000  # DCAD maxRecordCount = 4000
    while offset < total:
        params = {
            'f': 'json',
            'where': where,
            'outFields': '*',
            'returnGeometry': 'true',
            'geometryPrecision': 6,
            'resultOffset': offset,
            'resultRecordCount': chunk,
        }
        resp = requests.get(DCAD_QUERY_URL, params=params, timeout=60)
        resp.raise_for_status()
        features = resp.json().get('features', [])
        for feat in features:
            attrs = feat['attributes']
            geom = feat.get('geometry')
            if geom and 'rings' in geom and geom['rings']:
                try:
                    attrs['geometry'] = Polygon(geom['rings'][0])
                except Exception:
                    attrs['geometry'] = None
            else:
                attrs['geometry'] = None
            all_features.append(attrs)
        print(f'  Fetched {offset}–{offset + len(features)}')
        if len(features) < chunk:
            break
        offset += chunk

    # DCAD returns coordinates in EPSG:3857 (Web Mercator)
    result = gpd.GeoDataFrame(all_features, crs='EPSG:3857')
    result = result[result['geometry'].notna()].copy()
    result = result.to_crs('EPSG:4326')
    return result


if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f'Loaded {len(gdf):,} parcels from cache')
else:
    gdf = _download_dcad_addison()
    gdf.to_parquet(PARCEL_PATH)
    print(f'Downloaded and cached {len(gdf):,} parcels')

print(f'\nParcel count: {len(gdf):,}')
print(f'Columns: {[c for c in gdf.columns if c != "geometry"]}')
print(gdf[['LNDVALUE', 'IMPVALUE', 'CNTASSDVAL']].describe())

Loaded 3,301 parcels from cache

Parcel count: 3,301
Columns: ['OBJECTID', 'LOWPARCELID', 'PARCELID', 'BUILDING', 'UNIT', 'STATEDAREA', 'LGLSTARTDT', 'CVTTXCD', 'CVTTXDSCRP', 'SCHLTXCD', 'SCHLDSCRP', 'USECD', 'USEDSCRP', 'NGHBRHDCD', 'CLASSCD', 'CLASSDSCRP', 'SITEADDRESS', 'PRPRTYDSCRP', 'CNVYNAME', 'OWNERNME1', 'OWNERNME2', 'PSTLADDRESS', 'PSTLCITY', 'PSTLSTATE', 'PSTLZIP5', 'PSTLZIP4', 'FLOORCOUNT', 'BLDGAREA', 'RESFLRAREA', 'RESYRBLT', 'RESSTRTYP', 'STRCLASS', 'CLASSMOD', 'LNDVALUE', 'PRVASSDVAL', 'CNTASSDVAL', 'LASTUPDATE', 'MAPGRID', 'REVALYR', 'PRVRVALYR', 'IMPVALUE', 'DBA1', 'PRPSDVAL', 'Shape.STArea()', 'Shape.STLength()']
           LNDVALUE      IMPVALUE    CNTASSDVAL
count  3.301000e+03  3.301000e+03  3.301000e+03
mean   3.673927e+05  1.151437e+06  1.518830e+06
std    2.092791e+06  6.743573e+06  7.788178e+06
min    0.000000e+00  0.000000e+00  1.000000e+02
25%    7.279000e+04  1.131200e+05  2.408200e+05
50%    1.000000e+05  3.860400e+05  4.852800e+05
75%    1.200000e+05  4.95

## Section 3 — Classify and Validate

In [3]:
# Clip negative values
gdf['LNDVALUE'] = gdf['LNDVALUE'].fillna(0).clip(lower=0)
gdf['IMPVALUE'] = gdf['IMPVALUE'].fillna(0).clip(lower=0)
gdf['CNTASSDVAL'] = gdf['CNTASSDVAL'].fillna(0).clip(lower=0)

# Full exemption flag: government, school, religious owners
owner = gdf['OWNERNME1'].fillna('').str.upper()
gdf['full_exmp'] = owner.apply(
    lambda o: int(any(k in o for k in EXEMPT_OWNER_KEYWORDS))
)

# Homestead approximation: apply 20% city exemption to likely owner-occupied residential
gdf['hs_approx'] = gdf['CLASSCD'].isin(HOMESTEAD_CLASSES).astype(int)

# Property category mapping using DCAD class codes
def classify_parcel(row):
    cls = str(row.get('CLASSCD', '') or '').strip()
    if cls == '1':  return 'Single Family Residential'
    if cls == '2':  return 'Townhome / Rowhouse'
    if cls == '3':  return 'Condominium'
    if cls == '5':  return 'Large Multi-Family (5+ units)'
    if cls == '6':  return 'Small Multi-Family (2-4 units)'
    if cls in ('7', '8', '9', '10', '39'):  return 'Vacant Land'
    if cls == '11': return 'Other'
    if cls == '17': return 'Commercial'
    if cls == '18': return 'Industrial'
    if cls == '24': return 'Other'     # electric utility
    if cls == '35': return 'Other Residential'  # mobile home on leased land
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)

# Override: $0 improvement → Vacant Land (regardless of class code)
gdf.loc[gdf['IMPVALUE'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f'\nFull-exempt parcels: {gdf["full_exmp"].sum()}')
print(f'Homestead-approx parcels: {gdf["hs_approx"].sum()}')
print(f'Total parcels: {len(gdf):,}')

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential         1215
Condominium                        775
Vacant Land                        451
Townhome / Rowhouse                346
Commercial                         341
Small Multi-Family (2-4 units)      87
Large Multi-Family (5+ units)       75
Other Residential                    6
Industrial                           4
Other                                1
Name: count, dtype: int64

Full-exempt parcels: 93
Homestead-approx parcels: 2424
Total parcels: 3,301


## Section 4 — Current Tax Model

Texas levies at 100% market value. No fractional assessment ratio.

**Rate**: $0.6081 per $100 assessed value (FY2025 ACFR confirmed; same rate adopted for FY2026)  
**Homestead exemption**: 20% of market value for owner-occupied residential (city levy)  

**Validation against FY2025 ACFR**:  
- ACFR real property taxable: $5,490.8M → implied levy ~$33.4M  
- This model taxable base: $4,501.3M → modeled levy $27.4M (~18% gap)  
- Gap attributed to `PSTLCITY` filter missing border parcels + data vintage (DCAD = Jan 2025 values; ACFR = Jan 2024 values)  
- BPP ($968.2M taxable, ~$5.9M levy) excluded from this model by design

In [4]:
# Taxable values: apply homestead exemption to qualifying residential parcels
# Scale = 0.80 for homestead, 1.00 for all others
hs_scale = np.where(gdf['hs_approx'] == 1, 1.0 - HOMESTEAD_RATE, 1.0)
gdf['taxable_land_value'] = gdf['LNDVALUE'] * hs_scale
gdf['taxable_improvement_value'] = gdf['IMPVALUE'] * hs_scale

# Zero out fully-exempt parcels
gdf.loc[gdf['full_exmp'] == 1, 'taxable_land_value'] = 0.0
gdf.loc[gdf['full_exmp'] == 1, 'taxable_improvement_value'] = 0.0

# Current tax: city levy only
# MILLAGE_ADDISON is per $100; divide by 100 to get per-$ rate
gdf['current_tax'] = np.where(
    gdf['full_exmp'] == 1,
    0.0,
    (gdf['taxable_land_value'] + gdf['taxable_improvement_value']) * MILLAGE_ADDISON / 100.0
)

current_revenue = float(gdf['current_tax'].sum())
taxable_base = (gdf['taxable_land_value'] + gdf['taxable_improvement_value']).sum()

# FY2025 ACFR reference figures
ACFR_TOTAL_LEVY       = 39_493_427
ACFR_REAL_PROP_TAX    = 33_388_382   # $5,490.8M × 0.6081%
ACFR_BPP_LEVY         =  5_887_671   # $968.2M × 0.6081%

print(f'Modeled city real-property revenue:  ${current_revenue:>15,.0f}')
print(f'ACFR FY2025 real-property levy:      ${ACFR_REAL_PROP_TAX:>15,.0f}')
print(f'Gap vs. ACFR real property:          {(current_revenue/ACFR_REAL_PROP_TAX - 1)*100:>+14.1f}%')
print()
print(f'Taxable base (after HS exemption):   ${taxable_base/1e6:>14.1f}M')
print(f'ACFR real property taxable:          ${5_490_752_180/1e6:>14.1f}M')
print()
print(f'ACFR total levy (real + BPP):        ${ACFR_TOTAL_LEVY:>15,.0f}')
print(f'  Real property:                     ${ACFR_REAL_PROP_TAX:>15,.0f}')
print(f'  BPP (excluded from this model):    ${ACFR_BPP_LEVY:>15,.0f}')

Modeled city real-property revenue:  $     27,372,527
ACFR FY2025 real-property levy:      $     33,388,382
Gap vs. ACFR real property:                   -18.0%

Taxable base (after HS exemption):   $        4501.3M
ACFR real property taxable:          $        5490.8M

ACFR total levy (real + BPP):        $     39,493,427
  Real property:                     $     33,388,382
  BPP (excluded from this model):    $      5,887,671


## Section 5 — Split-Rate Model (4:1)

In [5]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
print(f'Solving on {len(taxable):,} taxable parcels (excluded {gdf["full_exmp"].sum()} fully-exempt)')

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=float(taxable['current_tax'].sum()),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine with exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} per $1,000')
print(f'Improvement millage: {improvement_millage:.4f} per $1,000')
print(f'Land/improvement ratio: {land_millage/improvement_millage:.2f}:1')
print(f'Revenue check: ${new_revenue:,.0f} (target ${taxable["current_tax"].sum():,.0f})')

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Addison TX — 4:1 Split-Rate Tax Impact')

Solving on 3,208 taxable parcels (excluded 93 fully-exempt)
Land millage:        14.8329 per $1,000
Improvement millage: 3.7082 per $1,000
Land/improvement ratio: 4.00:1
Revenue check: $27,372,527 (target $27,372,527)

Addison TX — 4:1 Split-Rate Tax Impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential   1215        $219,396        6.2%       $181         $-65    2.0%      -2.4%            10.2%             8.0%
                   Condominium    775        $-82,435       -7.6%      $-106          $-1   12.0%      -0.6%            48.1%            39.2%
                   Vacant Land    451        $923,090      143.9%     $2,047         $875  123.5%     143.9%            85.8%             0.0%
           Townhome / Rowhouse    346        $-48,289       -5.3%      $-140         $-93   -4.8%      -4.0%             0.3%            17.6%
                    Commerc

## Validation Summary

| Check | Result |
|---|---|
| Rate | $0.6081/$100 confirmed via FY2025 ACFR |
| Revenue match (real property) | Modeled $27.4M vs. ACFR $33.4M (−18%). Gap = missing border parcels (PSTLCITY filter) + data vintage (DCAD Jan 2025 vs. ACFR Jan 2024 values) |
| BPP levy (excluded by design) | ~$5.9M; total ACFR levy $39.5M = $33.4M real + $5.9M BPP |
| Parcel count | 3,301 real property parcels (93 fully-exempt, 3,208 taxable) |
| Census coverage | 100.0% spatially joined; 98.9% (3,266/3,301) have median_income |
| PNGs generated | 7 of 7 |
| SFR median change | −2.4% |
| Vacant land median change | +143.9% |
| Large MFR median change | −17.5% |
| Known limitation | ~$989M taxable real property missing from model; fix = spatial boundary filter instead of PSTLCITY |

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | $27.4M modeled from real property at $0.6081/$100 (FY2026 rate). Total Addison levy ~$40.6M includes BPP not modeled here — no real-property-only official figure to compare |
| Parcel count | 3,301 real property parcels (93 fully-exempt, 3,208 taxable) |
| Census coverage | 100.0% spatially joined; 98.9% (3,266/3,301) have median_income |
| PNGs generated | 7 of 7 |
| SFR median change | -2.4% |
| Vacant land median change | +143.9% |
| Large MFR median change | -17.5% |

In [6]:
fig, ax = plt.subplots(figsize=(10, 6))
summary = gdf[gdf['full_exmp'] == 0].groupby('PROPERTY_CATEGORY')['tax_change_pct'].median()
summary.sort_values().plot.barh(ax=ax)
ax.set_title('Addison TX — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('Saved category_preview.png')

Saved category_preview.png


## Section 7 — Census Join + Export

In [7]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf already carries demographics — spatial join adds them above.
            # Do NOT do a second gdf.merge(census_data) here: census_gdf has the columns
            # baked in, so a second merge creates median_income_x/y duplicates and silently
            # zeros out all demographic output.
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [8]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print('Done.')

  ✓ addison: 3,301 rows → ../../analysis/data/addison.csv  [model: split_rate_4to1]


Done.
